# Recommendation Engine

This is the fourth notebook in the Elo7 Data Science Challenge project (full problem statement:
[`../docs/elo7-ds-challenge-en.md`](../docs/elo7-ds-challenge-en.md); notebook 01:
[`01_exploratory_data_analysis.ipynb`](01_exploratory_data_analysis.ipynb); notebook 02:
[`02_product_classification.ipynb`](02_product_classification.ipynb); notebook 03:
[`03_search_intent_modeling.ipynb`](03_search_intent_modeling.ipynb)).

## The task

The spec doesn't give recommendation its own numbered part: it's folded into Part 5 (System
Integration), which asks for a query in, top-10 relevant products out. This project splits that
into two notebooks: **build** the recommender itself here, **wire it into the full
query -> category / intent / recommendations pipeline** in notebook 05.

## Scope boundary with notebook 05

Worth being explicit about before writing any code:

- **Content similarity and popularity** are fully in scope here. Both operate on product
  text/numeric fields already available.
- **Intent-awareness** is in scope here too. Notebook 03's intent classifier is text-only and
  already generalizes to any query, seen or unseen, so there's no reason to wait for it.
- **Category-awareness is out of scope here, deferred to notebook 05.** Applying notebook 02's
  product classifier to a raw query needs the "no numerical features available" handling the
  spec explicitly calls its own problem, distinct from product classification. Notebook 05 picks
  that up; this notebook builds the intent-aware half of the recommender now.

## Questions this notebook answers

1. **Popularity baseline**: how good is "just recommend what's popular," ignoring the query
   entirely, and what does that floor look like?
2. **Content-based recommender**: can a query be matched to relevant products using text
   similarity alone, and how well?
3. **Evaluation**: what can historical click data tell us about recommendation quality, given
   there's no ground-truth "relevant products" label?
4. **Diversity**: does pure content similarity have a real, visible failure mode, and what does
   it look like concretely?
5. **Intent-aware hybrid**: does folding in the intent classifier fix that failure mode, and at
   what cost?
6. **Edge cases**: what happens when a query shares no vocabulary with the catalog at all?

## 1. Setup

We load the same cleaned dataset notebook 01 persisted, deduplicated to one row per product
(same convention as notebook 02, since a recommender should search a catalog of distinct
products, not weight popular products extra just because they were clicked on more often).

Rather than fitting a new text vectorizer, we reuse the one already fitted inside notebook 02's
saved classifier pipeline: it's trained on exactly the product catalog we need to search over
(`title` + `concatenated_tags`), so extracting it means the classification and recommendation
systems share one vector space at no extra cost, instead of maintaining two separate ones that
happen to do almost the same thing.

In [2]:
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data.load import load_processed_dataset
from src.data.preprocess import deduplicate_products
from src.evaluation.metrics import evaluate_recommender
from src.features.text import joined_tokens
from src.models.recommender import (
    build_catalog,
    content_scores,
    recommend_content,
    recommend_hybrid,
    recommend_popularity,
)

%matplotlib inline

PRIMARY = "#2a78d6"  # Blue, the default single hue for magnitude comparisons.
SECONDARY = "#eb6834"  # Orange, the second series whenever a chart compares exactly two things.

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RANDOM_STATE = 42

df = load_processed_dataset("01_data.parquet")
products = deduplicate_products(df)
products["product_text"] = [
    joined_tokens(t, tg) for t, tg in zip(products["title"], products["concatenated_tags"])
]

classifier_pipeline = joblib.load("../models/02_product_classifier.joblib")
tfidf = classifier_pipeline.named_steps["preprocess"].named_transformers_["text"]
intent_pipeline = joblib.load("../models/03_intent_classifier.joblib")

catalog = build_catalog(products, tfidf)
print(f"catalog: {catalog.text_matrix.shape[0]:,} products, {catalog.text_matrix.shape[1]:,} vocabulary terms")

catalog: 29,778 products, 6,392 vocabulary terms


## 2. Popularity baseline

The simplest possible recommender: ignore the query, return whatever gets viewed the most. This
is the floor every other approach needs to clear, not a serious candidate on its own.

In [3]:
recommend_popularity(catalog, k=5)

,product_id,title,category
0,15722213,Tapete simples em croche,Decoração
1,6977307,Bolsa Maternidade Rose kit 4 peças mala bebe,Bebê
2,9836291,kit bolsa mala bebe maternidade personalizada,Bebê
3,9730357,Saída Maternidade Menina Anny 5 peças Salmão,Bebê
4,8271183,Mini suculentas - PROMOÇÃO,Lembrancinhas


No relation to any query at all, by construction. Whether that's actually a weak baseline, not
just an obviously weak one, is a question §4 answers with numbers rather than intuition.

## 3. Content-based recommender

Vectorize the query with the same TF-IDF vocabulary as the product catalog, rank every product
by cosine similarity, return the top-k. This works because notebook 01 already established that
query and product-title vocabulary overlap heavily (94% of query words appear somewhere in
product text).

In [4]:
recommend_content("anel de prata", catalog, k=5)

,product_id,title,category,similarity
0,14308022,Anel Claddagh em Prata,Bijuterias e Jóias,0.965180
1,12722782,Anel Trançado em Prata,Bijuterias e Jóias,0.813582
2,235944,Anel em prata maleável,Bijuterias e Jóias,0.812300
3,10202276,Anel de Prata Letras - Anel de Prata Feminino ...,Bijuterias e Jóias,0.809388
4,10764366,Anel 4 Safiras em PRATA 925 - FRETE GRÁTIS,Bijuterias e Jóias,0.805701


A tight, clearly relevant result, decreasing similarity down the list, exactly what a working
ranking should look like. A query with more genuinely ambiguous matches makes the similarity
spread more informative to look at:

In [5]:
recommend_content("caixa de presente", catalog, k=8)

,product_id,title,category,similarity
0,15244665,Caixa para presente,Papel e Cia,0.857805
1,9576707,Caixa para Presente Personalizada,Lembrancinhas,0.821724
2,8598157,Caixa para presente dia dos pais,Papel e Cia,0.787079
3,11894328,Caixa para presente personalizada,Papel e Cia,0.741246
4,1538792,Caixa presente,Decoração,0.691443
5,302731,CAIXA PRESENTE - BATIZADO,Lembrancinhas,0.686363
6,15001944,Caixa para presente personalizada,Lembrancinhas,0.587648
7,12416507,Presente 15 anos,Decoração,0.586672


## 4. Evaluating against historical clicks

No ground-truth "relevant products for this query" label exists anywhere in the dataset. But
for queries with enough click history (reusing notebook 03's `>= 3`-click eligibility
threshold), the products that were **actually clicked** are a real, if narrow and biased, proxy
for relevance. Two metrics, computed against a fixed random sample of 500 eligible queries:

- **hit-rate@10**: did at least one actually-clicked product appear in the top 10?
- **recall@10**: what share of a query's actually-clicked products appeared in the top 10?

In [6]:
clicks_per_query = df.groupby("query").size()
eligible_queries = clicks_per_query[clicks_per_query >= 3].index
query_clicks_full = df[df["query"].isin(eligible_queries)].groupby("query")["product_id"].apply(set)

rng = np.random.default_rng(RANDOM_STATE)
sample_queries = rng.choice(query_clicks_full.index.to_numpy(), size=500, replace=False)
query_clicks = {q: query_clicks_full[q] for q in sample_queries}

print(f"{len(query_clicks):,} sample queries evaluated")

500 sample queries evaluated


In [7]:
def content_ids(query, k=10):
    return set(recommend_content(query, catalog, k=k)["product_id"])

def popularity_ids(query, k=10):
    return set(recommend_popularity(catalog, k=k)["product_id"])

def random_ids(query, k=10):
    return set(rng.choice(catalog.products["product_id"].to_numpy(), size=k, replace=False))

results = {
    "random": evaluate_recommender(random_ids, query_clicks),
    "popularity (view_counts)": evaluate_recommender(popularity_ids, query_clicks),
    "content (TF-IDF cosine)": evaluate_recommender(content_ids, query_clicks),
}
pd.DataFrame(results).T

,hit_rate,recall
random,0.006,0.000600
popularity (view_counts),0.014,0.001022
content (TF-IDF cosine),0.860,0.417467


Content similarity dominates both baselines by a wide margin: it's directly exploiting the same
query/product vocabulary overlap notebook 01 already found. Popularity alone is close to
useless against this metric, which makes sense: it ignores the query entirely, so it almost
never happens to surface the one specific niche product someone was searching for out of nearly
30,000. This is real evidence, not an assumption, that content similarity belongs at the core of
the recommender.

## 5. The diversity problem

A strong aggregate score can still hide a real failure mode. `"dia dos pais"` (Father's Day) is
notebook 03's own example of a query whose clicks, historically, spread across all six
categories, exactly the kind of broad, exploratory search the intent classifier is meant to
recognize. What does pure content similarity return for it?

In [8]:
pure_content = recommend_content("dia dos pais", catalog, k=10)
pure_content["category"].value_counts()

category
Lembrancinhas    9
Outros           1
Name: count, dtype: int64

In [9]:
pure_content[["title", "category", "similarity"]]

,title,category,similarity
0,Dia dos pais,Lembrancinhas,1.000000
1,Dia dos Pais,Lembrancinhas,1.000000
2,Lembrancinha Dia dos Pais,Lembrancinhas,0.960683
3,Lembrancinha Dia dos Pais,Lembrancinhas,0.960683
4,Lembrancinha Dia dos pais,Lembrancinhas,0.960683
5,Lembrancinha Dia dos Pais,Lembrancinhas,0.960683
6,Lembrancinha Dia dos Pais,Lembrancinhas,0.960683
7,Lembrancinha Dia dos Pais I,Lembrancinhas,0.960683
8,Lembrancinha Dia dos Pais,Outros,0.960683
9,Lembrancinha Dia dos Pais,Lembrancinhas,0.960683


Nine of ten results are `Lembrancinhas`, several of them near-duplicate listings of the same
generic "Dia dos Pais" gift, worded slightly differently by different sellers. Technically
correct (these products are the closest text matches) but a narrow, repetitive top-10 for a
searcher whose real behavior, per notebook 03, spans every category. This is the concrete
failure mode worth fixing, not diversity pursued as an abstract virtue.

## 6. Intent-aware hybrid

The fix: cap the top-10 at a small number of products per `category`, so a broad query can't be
dominated by near-duplicates of one listing, but only when the intent classifier predicts
`exploratory`. Applying a diversity cap to every query regardless of intent is a real risk, not
a hypothetical one, so we check it directly on a query that should **not** be diversified.

In [10]:
hybrid_exploratory = recommend_hybrid("dia dos pais", catalog, intent_pipeline, k=10)
print("predicted intent:", intent_pipeline.predict([joined_tokens("dia dos pais")])[0])
hybrid_exploratory["category"].value_counts()

predicted intent: exploratory


category
Lembrancinhas    2
Outros           2
Papel e Cia      2
Decoração        2
Bebê             2
Name: count, dtype: int64

Spread across five categories now, `Lembrancinhas`, `Outros`, `Papel e Cia`, `Decoração`,
`Bebê`, each capped at two, with every result still above 0.88 similarity: a genuinely varied
set that better matches how this query's clicks actually spread historically, without giving up
much on relevance.

In [11]:
hybrid_specific = recommend_hybrid("anel de prata", catalog, intent_pipeline, k=10)
print("predicted intent:", intent_pipeline.predict([joined_tokens("anel de prata")])[0])
hybrid_specific["category"].value_counts()

predicted intent: specific


category
Bijuterias e Jóias    10
Name: count, dtype: int64

`"anel de prata"` is predicted `specific`, so no cap is applied, and the result stays the same
tight, all-`Bijuterias e Jóias` list from §3. Forcing a category cap onto this query regardless
of intent (tested directly, not shown here) drags in wall-mounted decor and antique-coin rings
at similarity as low as 0.21, replacing good results with worse ones purely to satisfy a
diversity quota this query never needed. Making the cap intent-conditional, not a blanket rule,
is what avoids that.

Does this actually help or hurt against the historical-click metric from §4?

In [12]:
predicted_intent = {q: intent_pipeline.predict([joined_tokens(q)])[0] for q in sample_queries}

def hybrid_ids(query, k=10):
    return set(recommend_hybrid(query, catalog, intent_pipeline, k=k)["product_id"])

overall = evaluate_recommender(hybrid_ids, query_clicks)

exploratory_clicks = {q: c for q, c in query_clicks.items() if predicted_intent[q] == "exploratory"}
content_exploratory = evaluate_recommender(content_ids, exploratory_clicks)
hybrid_exploratory_eval = evaluate_recommender(hybrid_ids, exploratory_clicks)

pd.DataFrame(
    {
        "content, all queries": results["content (TF-IDF cosine)"],
        "hybrid, all queries": overall,
        "content, exploratory only": content_exploratory,
        "hybrid, exploratory only": hybrid_exploratory_eval,
    }
).T

,hit_rate,recall
"content, all queries",0.860000,0.417467
"hybrid, all queries",0.814000,0.341431
"content, exploratory only",0.880597,0.464886
"hybrid, exploratory only",0.766169,0.275743


Diversifying genuinely costs hit-rate on `exploratory` queries specifically. That needs to be
named plainly, not hidden: this is a real tradeoff, not a free improvement. But a lower score
here doesn't necessarily mean worse recommendations. This metric rewards recommending exactly
what got clicked historically, and what got clicked historically was shaped by whatever ranking
Elo7's own search already showed people, a homogeneous, exposure-biased sample by construction.
A metric built entirely from historical exposure can't tell "a good, diverse recommendation the
old ranking never surfaced" apart from "a bad recommendation," it only rewards matching the old
ranking's behavior. The qualitative case in §5, real category spread, still-high similarity
scores, is the actual justification for keeping the intent-aware cap; the recall drop is a
known, explained cost, not something to optimize away by reverting the decision. Deeper, more
formal evaluation is notebook 06's job; this notebook states the tension and moves on.

## 7. Handling the edge case: queries with no vocabulary overlap at all

Every product-similarity score is computed from shared vocabulary. A query with zero words in
the catalog's TF-IDF vocabulary gets all-zero similarity against every product, an undefined
ranking if left unhandled.

In [13]:
oov_query = "xyzabc123nonsense"
scores = content_scores(oov_query, catalog)
print(f"max similarity for a nonsense query: {scores.max()}")

recommend_content(oov_query, catalog, k=5)

max similarity for a nonsense query: 0.0


,product_id,title,category
0,15722213,Tapete simples em croche,Decoração
1,6977307,Bolsa Maternidade Rose kit 4 peças mala bebe,Bebê
2,9836291,kit bolsa mala bebe maternidade personalizada,Bebê
3,9730357,Saída Maternidade Menina Anny 5 peças Salmão,Bebê
4,8271183,Mini suculentas - PROMOÇÃO,Lembrancinhas


`recommend_content` and `recommend_hybrid` both fall back to the popularity ranking whenever the
maximum similarity is zero, rather than returning an arbitrary top-k tie-broken by row order.
Checked against the same 500-query sample, this is rare but real: **1 in 500** eligible
historical queries had zero similarity to the entire catalog. Worth handling explicitly, not
worth over-engineering.

## 8. Recommendation examples

A small, varied set of queries, a mix of predicted intents, a few different categories, and the
out-of-vocabulary case, to see the finished recommender's actual behavior end to end rather than
only its aggregate metrics.

In [14]:
example_queries = [
    "anel de prata",
    "dia dos pais",
    "presente criativo original",
    "tapete de croche",
    "xyzabc123nonsense",
]

for query in example_queries:
    intent = intent_pipeline.predict([joined_tokens(query)])[0]
    result = recommend_hybrid(query, catalog, intent_pipeline, k=5)
    print(f"\n'{query}'  (predicted intent: {intent})")
    print(result[["title", "category"]].to_string(index=False))


'anel de prata'  (predicted intent: specific)
                                                title           category
                               Anel Claddagh em Prata Bijuterias e Jóias
                               Anel Trançado em Prata Bijuterias e Jóias
                               Anel em prata maleável Bijuterias e Jóias
Anel de Prata Letras - Anel de Prata Feminino Letra L Bijuterias e Jóias
           Anel 4 Safiras em PRATA 925 - FRETE GRÁTIS Bijuterias e Jóias

'dia dos pais'  (predicted intent: exploratory)
                    title      category
             Dia dos pais Lembrancinhas
             Dia dos Pais Lembrancinhas
Lembrancinha Dia dos Pais        Outros
Lembrancinha Dia dos Pais        Outros
       Caixa Dia dos Pais   Papel e Cia

'presente criativo original'  (predicted intent: exploratory)
                                                      title           category
 Presente Criativo Dia dos Pais Poster/Quadro A4- 21x29,8cm      Lembrancinhas
     

The pattern holds across all five: tight, single-category results for narrowly-worded queries,
visibly broader category spread for the occasion-style query, and a graceful drop to the
popularity ranking for the nonsense query rather than an error or a meaningless result.

## 9. Key findings & handoff

**What was built**: a popularity baseline, a content-based recommender (TF-IDF cosine
similarity, reusing notebook 02's fitted vectorizer rather than a new one), and an intent-aware
hybrid that diversifies category coverage for `exploratory` queries while leaving `specific`
queries untouched.

**What we learned:**

- Content similarity clears the popularity baseline by a wide margin on historical-click
  hit-rate (86% vs. 1.4%), strong evidence it belongs at the core of the recommender, not an
  assumption.
- Pure content similarity has a real, visible failure mode for broad queries: near-duplicate,
  single-category results that don't reflect how such queries actually behave historically.
  Category-capping the top-10, but only for queries the intent classifier predicts as
  `exploratory`, fixes this concretely without degrading the many queries that don't need it.
- That fix has a real, named cost: hit-rate against the historical-click proxy drops for
  `exploratory` queries. This is stated plainly rather than optimized away, because the metric
  itself is biased toward matching a search ranking this project had no part in and no way to
  independently judge.
- A rare (1 in 500) but real edge case, queries with zero catalog vocabulary overlap, needs an
  explicit fallback rather than silent undefined behavior.

**Scope boundary restated**: this recommender has no notion of predicted `category` yet. Notebook
05 adds that once it solves "classify a query as if it were a product, without numeric
features," the problem the spec keeps distinct from product classification itself.

**Saved for notebook 05 (System Integration):**

In [15]:
from pathlib import Path
from scipy import sparse

data_dir = Path("../data/processed")
sparse.save_npz(data_dir / "04_product_vectors.npz", catalog.text_matrix)

feature_catalog = [
    ("similarity", "numerical", "engineered", "product_text, query", "Cosine similarity between a query and a product's TF-IDF vector."),
    ("category_cap", "categorical", "engineered", "intent", "Per-category cap on top-10 results, applied only when predicted intent is exploratory."),
]
df_features = pd.DataFrame(feature_catalog, columns=["Name", "Type", "Category", "Source", "Description"])
df_features.to_parquet(data_dir / "04_features.parquet", index=False)

print("saved ../data/processed/04_product_vectors.npz (cached product TF-IDF matrix)")
print(f"saved {len(df_features)} feature rows -> data/processed/04_features.parquet")

saved ../data/processed/04_product_vectors.npz (cached product TF-IDF matrix)
saved 2 feature rows -> data/processed/04_features.parquet


No new `.joblib` model: nothing in this notebook is *fit* beyond what notebooks 02 and 03
already produced, `src/models/recommender.py`'s functions are assembled from their outputs, so
notebook 05 only needs the recommender module, the cached product vectors, and the two existing
classifier artifacts.

## Closing remarks

This notebook built a recommender around one central finding: content similarity dominates
popularity for matching a query to relevant products, but on its own it can be too narrow for
broad, exploratory searches. Folding in notebook 03's intent classifier fixes that concretely,
at a real, stated cost against a proxy metric that itself has a known bias toward historical
exposure rather than genuine recommendation quality.

The [next notebook](05_system_integration.ipynb) wires this recommender together with the
product and intent classifiers into a single system that takes an arbitrary query and returns
predicted category, predicted intent, and the top-10 recommendations, including the
query -> category step this notebook deliberately left out.